In [1]:
import pandas as pd
import numpy as np

import pandas as pd
import numpy as np

def stratified_sample_percentage(
    df,
    label_col,
    percentage,
    min_per_class=10,
    random_state=42):
    """
    Stratified sampling with minimum samples per class constraint.
    """

    if not (0 < percentage <= 100):
        raise ValueError("percentage must be in (0, 100]")

    frac = percentage / 100.0
    sampled_list = []

    for cls, group in df.groupby(label_col):
        n_original = len(group)

        # desired sample
        n_samples = int(round(n_original * frac))

        # enforce minimum
        n_samples = max(n_samples, min_per_class)

        # cannot exceed available data
        n_samples = min(n_samples, n_original)

        sampled_group = group.sample(n=n_samples, random_state=random_state)
        sampled_list.append(sampled_group)

    sampled_df = pd.concat(sampled_list).sample(frac=1, random_state=random_state).reset_index(drop=True)

    return sampled_df
    
# Path to your CSV file
file_path = "big.data/NF-UNSW-NB15-v3.csv"

# Read CSV
df = pd.read_csv(file_path)

# Print first 5 rows
print(df.head())

   FLOW_START_MILLISECONDS  FLOW_END_MILLISECONDS IPV4_SRC_ADDR  L4_SRC_PORT  \
0            1424242193040          1424242193043    59.166.0.2         4894   
1            1424242192744          1424242193079    59.166.0.4        52671   
2            1424242190649          1424242193109    59.166.0.0        47290   
3            1424242193145          1424242193146    59.166.0.8        43310   
4            1424242193239          1424242193241    59.166.0.1        45870   

   IPV4_DST_ADDR  L4_DST_PORT  PROTOCOL  L7_PROTO  IN_BYTES  IN_PKTS  ...  \
0  149.171.126.3           53        17       5.0       146        2  ...   
1  149.171.126.6        31992         6      11.0      4704       28  ...   
2  149.171.126.9         6881         6      37.0     13662      238  ...   
3  149.171.126.7           53        17       5.0       146        2  ...   
4  149.171.126.1           53        17       5.0       130        2  ...   

   SRC_TO_DST_IAT_MIN  SRC_TO_DST_IAT_MAX  SRC_TO_DST_IA

In [2]:
print(df['Attack'].unique())

['Benign' 'Fuzzers' 'Exploits' 'Backdoor' 'Reconnaissance' 'Generic' 'DoS'
 'Shellcode' 'Analysis' 'Worms']


In [3]:
df.columns

Index(['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR',
       'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO',
       'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS',
       'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS',
       'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT',
       'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
       'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES',
       'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS',
       'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS',
       'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
       'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES',
       'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
       'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
       'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE',
       'DNS_TTL_ANSWER', 'FTP_COMMAN

In [4]:
unique_df = pd.DataFrame({
    col: pd.Series(df[col].unique())
    for col in df.columns
})

unique_df

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1.424242e+12,1424242193043,59.166.0.2,4894.0,149.171.126.3,53.0,17.0,5.0,146.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,1.424242e+12,1424242193079,59.166.0.4,52671.0,149.171.126.6,31992.0,6.0,11.0,4704.0,28.0,...,10000.0,91.0,12.0,19.0,6.0,90.0,12.0,19.0,1.0,Fuzzers
2,1.424242e+12,1424242193109,59.166.0.0,47290.0,149.171.126.9,6881.0,89.0,37.0,13662.0,238.0,...,1.0,1843.0,10.0,119.0,10.0,1843.0,5.0,88.0,NaN,Exploits
3,1.424242e+12,1424242193146,59.166.0.8,43310.0,149.171.126.7,48138.0,1.0,0.0,130.0,6.0,...,41976.0,2.0,251.0,1.0,5.0,2.0,1.0,2.0,NaN,Backdoor
4,1.424242e+12,1424242193241,59.166.0.1,45870.0,149.171.126.1,30872.0,132.0,4.0,320.0,8.0,...,14497.0,4.0,85.0,355.0,79.0,1.0,60.0,1.0,NaN,Reconnaissance
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2299727,NaN,1421972723458,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2299728,NaN,1421972723766,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2299729,NaN,1421972723563,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2299730,NaN,1421972723759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df_clean = df.dropna(subset=["Label", "Attack"], how="all")

In [6]:
df_clean.head()

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1424242193040,1424242193043,59.166.0.2,4894,149.171.126.3,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
1,1424242192744,1424242193079,59.166.0.4,52671,149.171.126.6,31992,6,11.0,4704,28,...,0,91,12,19,0,90,12,19,0,Benign
2,1424242190649,1424242193109,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,...,0,1843,10,119,0,1843,5,88,0,Benign
3,1424242193145,1424242193146,59.166.0.8,43310,149.171.126.7,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
4,1424242193239,1424242193241,59.166.0.1,45870,149.171.126.1,53,17,5.0,130,2,...,0,0,0,0,0,0,0,0,0,Benign


In [7]:
cols_to_drop = [
    # leakage
    "Label",

    # identifiers
    "IPV4_SRC_ADDR", "IPV4_DST_ADDR",
    "L4_SRC_PORT", "L4_DST_PORT",
    "DNS_QUERY_ID",

    # timestamps
    "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS",

    # redundant duration
    "DURATION_IN", "DURATION_OUT",
]

df_new = df.drop(columns=cols_to_drop)

In [8]:
def sanitize_df(df, exclude_cols=None, clip_value=1e10):
    """
    Cleans a DataFrame:
    - converts to numeric (non-convertible → NaN)
    - replaces inf / -inf
    - fills NaN
    - clips extreme values

    Parameters:
    - df : pandas DataFrame
    - exclude_cols : list of columns to skip (e.g., label)
    - clip_value : max absolute value

    Returns:
    - cleaned DataFrame
    """

    df_clean = df.copy()

    if exclude_cols is None:
        exclude_cols = []

    for col in df_clean.columns:
        if col in exclude_cols:
            continue

        # Force numeric
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

        # Replace inf with NaN
        df_clean[col] = df_clean[col].replace([np.inf, -np.inf], np.nan)

        # Fill NaN (median is safer than 0)
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val if not np.isnan(median_val) else 0)

        # Clip extreme values
        df_clean[col] = df_clean[col].clip(-clip_value, clip_value)

    return df_clean

In [9]:
df_new = df_new.dropna()
df_new.isna().any().any()

np.False_

In [10]:
df_new = sanitize_df(df_new, exclude_cols=["Attack"])
label_col = "Attack"
y = df_new[label_col]

X_df = df_new.drop(columns=[label_col])

# Convert ONLY features
X_df = X_df.apply(pd.to_numeric, errors='coerce')

# Recombine
df_new = pd.concat([X_df, y], axis=1)

df_final = stratified_sample_percentage(
    df_new,
    label_col="Attack",
    percentage=0.1,
    min_per_class=10
)

df_final.count()

PROTOCOL                       2346
L7_PROTO                       2346
IN_BYTES                       2346
IN_PKTS                        2346
OUT_BYTES                      2346
OUT_PKTS                       2346
TCP_FLAGS                      2346
CLIENT_TCP_FLAGS               2346
SERVER_TCP_FLAGS               2346
FLOW_DURATION_MILLISECONDS     2346
MIN_TTL                        2346
MAX_TTL                        2346
LONGEST_FLOW_PKT               2346
SHORTEST_FLOW_PKT              2346
MIN_IP_PKT_LEN                 2346
MAX_IP_PKT_LEN                 2346
SRC_TO_DST_SECOND_BYTES        2346
DST_TO_SRC_SECOND_BYTES        2346
RETRANSMITTED_IN_BYTES         2346
RETRANSMITTED_IN_PKTS          2346
RETRANSMITTED_OUT_BYTES        2346
RETRANSMITTED_OUT_PKTS         2346
SRC_TO_DST_AVG_THROUGHPUT      2346
DST_TO_SRC_AVG_THROUGHPUT      2346
NUM_PKTS_UP_TO_128_BYTES       2346
NUM_PKTS_128_TO_256_BYTES      2346
NUM_PKTS_256_TO_512_BYTES      2346
NUM_PKTS_512_TO_1024_BYTES  

In [11]:
df_final.isna().any().any()

np.False_

In [12]:
df_final.to_csv("small.data/attack.csv", index=False)